In [10]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [11]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.8,
)

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

In [5]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
Userseeks factual information about a fictional moon capital (Lunapolis), including its weather, population of cheese miners, and the likelihood of a union strike.

## SUMMARY
- The moon capital is identified as Lunapolis.  
- Weather in Lunapolis: clear skies, temperature range from a high of 120 °C to a low of –100 °C.  
- Number of cheese miners living in Lunapolis: 100,000.  
- The cheese miners' union is predicted to strike because miners are unhappy with the new president.  
No alternative explanations or rejected options were presented.

## ARTIFACTS
None

## NEXT STEPS
No additional tasks remain; the context extracted above captures all relevant information for continuing the session.


## Trim/delete messages

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

@before_agent
def print_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Print all the messages in the state"""
    messages = state["messages"]

    print("Current messages in state:")
    for m in messages:
        print(f"{m.type}: {m.content}")

In [ ]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[print_messages, trim_messages, print_messages],
)

In [8]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='5fa279c9-1f89-42c5-a611-87f9b1681265'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='5d0cdd58-6d57-4079-af89-4a5348107c59', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='053cb0db-584f-4a17-8bb7-e8d2e26c1e2a'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='d1956036-65d1-4f9d-b855-4caf8012f24e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='f542476d-c2ed-4eff-a7f3-1edad0f823f5'),
              AIMessage(content='I don’t have access to real‑time data about your device, so I 

In [9]:
print(response["messages"][-1].content)

I don’t have access to real‑time data about your device, so I can’t tell its temperature. You can check the temperature by looking at any built‑in indicators, using a temperature‑monitoring app (if the device supports it), or consulting the device’s manual or specifications for its typical operating temperature range. If you have a specific reading you’d like help interpreting, feel free to share it!
